# Deduplication

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-07-deduplication.ipynb)

Duplicates arrive from retries, replayed messages, late updates and dirty source systems. Deduplication is not one
operation. First decide what counts as **the same record** (the key). Then decide **which copy wins** (the rule).
Without a deterministic rule, the result can change between runs.

This chapter shows how to find duplicates, keep one row per key, and apply updates. It ends with practice questions
and answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW contacts AS
SELECT * FROM VALUES (1, 'ava@x.com'), (2, 'ben@x.com'), (3, 'ava@x.com'), (4, ' Ava@X.com ')
AS contacts(id, email)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW profile_updates AS
SELECT * FROM VALUES
  (1, 1, 'old@x.com', DATE '2023-01-01'),
  (2, 1, 'new@x.com', DATE '2023-06-01'),
  (3, 1, 'mid@x.com', DATE '2023-03-01'),
  (4, 2, 'b@x.com',   DATE '2023-02-01'),
  (5, 2, 'b2@x.com',  DATE '2023-02-01')
AS profile_updates(update_id, customer_id, email, updated_at)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW limits AS
SELECT * FROM VALUES (1, 1000), (2, 2000), (3, 1500), (4, 500), (5, 800), (6, 900), (7, 300), (8, 400)
AS limits(customer_id, credit_limit)
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW limit_updates AS
SELECT * FROM VALUES
  (1, 1200, DATE '2023-05-01'),
  (1, 1500, DATE '2023-08-01'),
  (4, 700,  DATE '2023-04-01'),
  (7, 350,  DATE '2023-02-01')
AS limit_updates(customer_id, credit_limit, updated_at)
""")

## 1. Find duplicates

`GROUP BY key HAVING COUNT(*) > 1` lists the keys that appear more than once. When the key is written
inconsistently, normalise it first. `' Ava@X.com '` and `'ava@x.com'` are the same mailbox.

In [ ]:
spark.sql("""
SELECT email, COUNT(*) AS copies
FROM contacts
GROUP BY email
HAVING COUNT(*) > 1
""").show()

spark.sql("""
SELECT lower(trim(email)) AS normalised_email, COUNT(*) AS copies
FROM contacts
GROUP BY lower(trim(email))
HAVING COUNT(*) > 1
""").show()

## 2. Keep one row per key

`profile_updates` is an append-only feed with several versions per customer. Keep the newest version with
`ROW_NUMBER`. Customer 2 has two versions with the same date, so `update_id` breaks the tie: the higher id wins.
The tie-breaker makes the choice repeatable.

In [ ]:
spark.sql("""
SELECT customer_id, update_id, email
FROM (
  SELECT customer_id, update_id, email,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC, update_id DESC) AS rn
  FROM profile_updates
)
WHERE rn = 1
ORDER BY customer_id
""").show()

### The GROUP BY MAX pitfall

A common alternative joins each customer to their maximum timestamp. It is correct only when that maximum is unique.
Customer 2 has two rows at the maximum, so this query returns both.

In [ ]:
spark.sql("""
SELECT p.customer_id, p.email
FROM profile_updates p
JOIN (
  SELECT customer_id, MAX(updated_at) AS latest
  FROM profile_updates
  GROUP BY customer_id
) m ON m.customer_id = p.customer_id AND m.latest = p.updated_at
ORDER BY p.customer_id, p.email
""").show()

## 3. Delete duplicates by keeping the survivors

Temporary views cannot be modified, so you cannot `DELETE` duplicates from one. The equivalent is to select the
rows to keep. A row survives when no strictly newer version of the same customer exists. On a real table, the same
predicate drives a `DELETE` (see the note on MERGE below).

In [ ]:
spark.sql("""
SELECT p.update_id, p.customer_id, p.email
FROM profile_updates p
WHERE NOT EXISTS (
  SELECT 1 FROM profile_updates k
  WHERE k.customer_id = p.customer_id
    AND (k.updated_at > p.updated_at
         OR (k.updated_at = p.updated_at AND k.update_id > p.update_id))
)
ORDER BY p.update_id
""").show()

## 4. Apply updates to a base table

To update a table from a feed, take the latest update per key and join it to the base table. Use `COALESCE` so keys
without an update keep their current value. Forgetting that fallback is how a whole column gets silently set to
NULL.

In [ ]:
spark.sql("""
WITH latest AS (
  SELECT customer_id, credit_limit
  FROM (
    SELECT customer_id, credit_limit,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC) AS rn
    FROM limit_updates
  )
  WHERE rn = 1
)
SELECT b.customer_id, COALESCE(l.credit_limit, b.credit_limit) AS credit_limit
FROM limits b
LEFT JOIN latest l ON l.customer_id = b.customer_id
ORDER BY b.customer_id
""").show()

## 5. On a table format that supports it

Delta Lake and Apache Iceberg tables can be changed in place, and the update above becomes one `MERGE`, run after
the source has been reduced to one row per key. This cell is a reference only, because it needs such a table:

```sql
MERGE INTO limits AS target
USING latest_limit_updates AS source
  ON target.customer_id = source.customer_id
WHEN MATCHED THEN UPDATE SET target.credit_limit = source.credit_limit
WHEN NOT MATCHED THEN INSERT (customer_id, credit_limit) VALUES (source.customer_id, source.credit_limit)
```

## Summary

- Define the key, then define a deterministic winner. Add a unique tie-breaker.
- Normalise keys before comparing them.
- "Delete duplicates" means "keep the survivors". Express it as a predicate you can check before you run it.
- Guard updates with a fallback so untouched keys keep their values.

## Practice questions

1. Which email addresses appear more than once after normalising with `lower(trim(...))`? Show the normalised email and the count.
2. Keep the **earliest** version per customer, breaking ties on the lower `update_id`. Show `customer_id`, `update_id` and `email`, sorted by `customer_id`.
3. How many rows of `profile_updates` are not survivors, that is, have a strictly newer version of the same customer?

## Answers

### Question 1

In [ ]:
q = """
SELECT lower(trim(email)) AS normalised_email, COUNT(*) AS copies
FROM contacts
GROUP BY lower(trim(email))
HAVING COUNT(*) > 1
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("ava@x.com", 3)]

### Question 2

In [ ]:
q = """
SELECT customer_id, update_id, email
FROM (
  SELECT customer_id, update_id, email,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at ASC, update_id ASC) AS rn
  FROM profile_updates
)
WHERE rn = 1
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 1, "old@x.com"), (2, 4, "b@x.com")]

### Question 3

In [ ]:
q = """
SELECT COUNT(*) AS not_survivors
FROM profile_updates p
WHERE EXISTS (
  SELECT 1 FROM profile_updates k
  WHERE k.customer_id = p.customer_id
    AND (k.updated_at > p.updated_at
         OR (k.updated_at = p.updated_at AND k.update_id > p.update_id))
)
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(3,)]